#  H₂ Photocatalytic Production Rate Prediction
### LightGBM + CatBoost Ensemble with Anti-Overfitting Design

**Predicts H₂ production rates (μmol/g·h) from photocatalytic glycerol reforming experiments.**

| Metric | Value |
|---|---|
| Test R² | **0.719** |
| Test RMSE | ~14,278 μmol/g·h |
| Train–Test Gap | 0.156 (overfitting controlled) |

> **Note:** The theoretical R² ceiling for this dataset is ~0.80 due to multi-lab noise across source publications.

---


##  Cell 1 — Install Dependencies

In [ ]:
!pip install lightgbm catboost scikit-learn pandas numpy matplotlib -q

##  Cell 2 — Upload Your Data

Upload your CSV file when prompted. The file should be named:
`2. s excel.xlsm - Data.csv`

Or rename it and update the `DATA_FILE` variable below.


In [ ]:
from google.colab import files
import io

print("Please upload your CSV data file...")
uploaded = files.upload()

# Get the filename from the upload
DATA_FILE = list(uploaded.keys())[0]
print(f"\n Uploaded: {DATA_FILE}")


## 🔧 Cell 3 — Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from catboost import CatBoostRegressor, Pool
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.model_selection import KFold, train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# ── Configuration ──
TARGET     = 'H2vproduction rate(micromol/g.h)'
TEST_SIZE  = 0.2
RANDOM_STATE = 42
SEEDS      = [42, 123, 456]    # 3 seeds for LightGBM ensemble
N_FOLDS    = 5

print(" Imports complete")


##  Cell 4 — Load & Inspect Data

In [ ]:
df = pd.read_csv(DATA_FILE)

print(f"Dataset shape : {df.shape}")
print(f"Target range  : {df[TARGET].min():.1f} – {df[TARGET].max():.1f}  μmol/g·h")
print(f"\nColumn list:")
for c in df.columns:
    miss = df[c].isna().sum()
    print(f"  {c:<45} missing: {miss:>4}  ({miss/len(df)*100:.1f}%)")


##  Cell 5 — Feature Definition

In [ ]:
CAT_COLS = [
    'Cocatalyst 1', 'Semiconductor 1', 'Semiconductor 2', 'Semiconductor 3',
    'Form', 'Structure', 'Light', 'Light Source',
    'Preparation of Semiconductor', 'Preparation of Photocatalyst',
]

NUM_COLS = [
    'Semiconductor 1 %', 'Semiconductor 2 %', 'Co-Catalyst wt',
    'Power (W)', 'Filter (nm)', 'Photocatalyst load (g/L)',
    'Semiconductor Calcination Temp©', 'Semiconductor Calcination Time (h)',
    'Photocatalyst Calcination Temp ©', 'Photocatalyst Calcination Time (h)',
    'Glycerol Concentration (v%)', 'Solution Volume (ml)',
    'pH', 'Reaction Temp (C)', 'Bandgap (eV)',
]

# Target encoding applied to low-cardinality categoricals only (prevents leakage)
TE_COLS = ['Cocatalyst 1', 'Semiconductor 1']

print(f"Categorical features : {len(CAT_COLS)}")
print(f"Numerical features   : {len(NUM_COLS)}")
print(f"Target-encoded cols  : {len(TE_COLS)}")


##  Cell 6 — Preprocessing

In [ ]:
# Store filter NaN mask before imputation
no_filter_mask = df['Filter (nm)'].isna()

# Fill categoricals with 'None', numericals with median
for col in CAT_COLS:
    df[col] = df[col].fillna('None').astype(str)
for col in NUM_COLS:
    df[col] = df[col].fillna(df[col].median())

print(f"Missing values after imputation:")
print(df[CAT_COLS + NUM_COLS].isnull().sum().sum(), "total remaining (should be 0)")
print(" Preprocessing complete")


## ⚗️ Cell 7 — Feature Engineering

**Design principle:** Only physically motivated interactions are kept.
High-cardinality combo features (e.g. `Cocatalyst × Semiconductor`) are intentionally
**excluded** to prevent target encoding leakage on small data (~909 rows).


In [ ]:
# ── Presence flags ──
df['has_cocatalyst']     = (df['Cocatalyst 1'] != 'None').astype(int)
df['has_semiconductor2'] = (df['Semiconductor 2'] != 'None').astype(int)
df['has_filter']         = (~no_filter_mask).astype(int)

# ── Physically motivated interactions ──
df['power_per_vol']   = df['Power (W)'] / (df['Solution Volume (ml)'] + 1)
df['load_x_glycerol'] = df['Photocatalyst load (g/L)'] * df['Glycerol Concentration (v%)']
df['bandgap_x_power'] = df['Bandgap (eV)'] * df['Power (W)']
df['wt_per_load']     = df['Co-Catalyst wt'] / (df['Photocatalyst load (g/L)'] + 1e-3)

# ── Log transforms for skewed numerics ──
df['log_power'] = np.log1p(df['Power (W)'])
df['log_load']  = np.log1p(df['Photocatalyst load (g/L)'])

ENG_COLS = [
    'has_cocatalyst', 'has_semiconductor2', 'has_filter',
    'power_per_vol', 'load_x_glycerol', 'bandgap_x_power', 'wt_per_load',
    'log_power', 'log_load',
]

FEATURES = CAT_COLS + NUM_COLS + ENG_COLS   # TE features appended after split

print(f"Engineered features  : {len(ENG_COLS)}")
print(f"Total features (pre-TE): {len(FEATURES)}")
print("Feature engineering complete")


##  Cell 8 — Log-Transform Target & Stratified Split

In [ ]:
# Log-transform to handle extreme skew in target
df['log_target'] = np.log1p(df[TARGET])

# Stratify by quantile so test set mirrors full target distribution
df['_quantile']  = pd.qcut(df['log_target'], q=5, labels=False)

X_all  = df[FEATURES].copy()
y_all  = df['log_target']
y_orig = df[TARGET]

X_train, X_test, y_train, y_test, yo_train, yo_test = train_test_split(
    X_all, y_all, y_orig,
    test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=df['_quantile']
)

print(f"Train : {len(X_train)} rows")
print(f"Test  : {len(X_test)} rows")

# Target distribution check
print(f"\nTrain target — mean: {yo_train.mean():.0f}  median: {yo_train.median():.0f}  max: {yo_train.max():.0f}")
print(f"Test  target — mean: {yo_test.mean():.0f}  median: {yo_test.median():.0f}  max: {yo_test.max():.0f}")


##  Cell 9 — OOF Target Encoding

Out-of-fold encoding on training data only → **zero leakage**.
Applied only to `Cocatalyst 1` (26 unique) and `Semiconductor 1` (31 unique).


In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

for col in TE_COLS:
    feat = f'te_{col}'
    oof  = np.full(len(X_train), y_train.mean())

    for tr_idx, val_idx in kf.split(X_train):
        te_map = y_train.iloc[tr_idx].groupby(X_train[col].iloc[tr_idx]).mean()
        oof[val_idx] = X_train[col].iloc[val_idx].map(te_map).fillna(y_train.mean()).values

    X_train[feat] = oof

    # Test: use all-train mean map
    global_map   = y_train.groupby(X_train[col]).mean()
    X_test[feat] = X_test[col].map(global_map).fillna(y_train.mean()).values
    FEATURES.append(feat)

print(f"Total features (with TE): {len(FEATURES)}")
for col in TE_COLS:
    print(f"  te_{col}  →  added")
print(" Target encoding complete (OOF, no leakage)")


##  Cell 10 — LightGBM: 5-Fold CV (3-seed ensemble)

**Key anti-overfitting settings vs naive baseline:**
- `num_leaves` 200 → **64**
- `min_data_in_leaf` 3 → **20**
- `lambda_l2` 0.05 → **1.0**
- Full-train refit uses **CV mean best_iteration** (not fixed 4000)


In [ ]:
lgb_base_params = {
    'objective':        'regression',
    'metric':           'rmse',
    'num_leaves':       64,
    'max_depth':        7,
    'min_data_in_leaf': 20,
    'feature_fraction': 0.70,
    'lambda_l1':        0.1,
    'lambda_l2':        1.0,
    'bagging_fraction': 0.80,
    'bagging_freq':     1,
    'verbose':          -1,
}
LGB_LR    = 0.02
LGB_BOOST = 3000
EARLY     = 200

# Prepare datasets with category dtype
X_tr_lgb = X_train[FEATURES].copy()
X_te_lgb = X_test[FEATURES].copy()
for c in CAT_COLS:
    X_tr_lgb[c] = X_tr_lgb[c].astype('category')
    X_te_lgb[c] = X_te_lgb[c].astype('category')

print("=" * 60)
print("  LightGBM — 5-Fold CV (3-seed ensemble)")
print("=" * 60)

fold_r2_lgb    = []
best_iters_lgb = []

for seed in SEEDS:
    params = {**lgb_base_params, 'learning_rate': LGB_LR, 'random_state': seed}
    seed_fold_r2  = []
    seed_best_its = []

    for fold, (tr, val) in enumerate(kf.split(X_tr_lgb), 1):
        Xtr, Xval = X_tr_lgb.iloc[tr], X_tr_lgb.iloc[val]
        ytr, yval = y_train.iloc[tr],  y_train.iloc[val]
        yov        = yo_train.iloc[val]

        td = lgb.Dataset(Xtr, ytr)
        vd = lgb.Dataset(Xval, yval, reference=td)
        m  = lgb.train(params, td, num_boost_round=LGB_BOOST, valid_sets=[vd],
                       callbacks=[lgb.early_stopping(EARLY, verbose=False),
                                  lgb.log_evaluation(-1)])
        p = np.expm1(m.predict(Xval))
        seed_fold_r2.append(r2_score(yov, p))
        seed_best_its.append(m.best_iteration)

    fold_r2_lgb.append(np.mean(seed_fold_r2))
    best_iters_lgb.append(int(np.mean(seed_best_its)))
    print(f"  Seed {seed:>4}: CV R²={np.mean(seed_fold_r2):.4f} ± {np.std(seed_fold_r2):.4f}  "
          f"best_iter≈{int(np.mean(seed_best_its))}")

mean_best_lgb = int(np.mean(best_iters_lgb))
print(f"\n  Mean CV best_iteration: {mean_best_lgb}")


##  Cell 11 — LightGBM: Full-Train Refit

In [ ]:
print(f"Retraining each seed on full train ({mean_best_lgb} iterations from CV)...")

lgb_test_final  = np.zeros(len(X_test))
lgb_train_final = np.zeros(len(X_train))
lgb_models      = []

for seed in SEEDS:
    params = {**lgb_base_params, 'learning_rate': LGB_LR, 'random_state': seed}
    m = lgb.train(params, lgb.Dataset(X_tr_lgb, y_train),
                  num_boost_round=mean_best_lgb,
                  callbacks=[lgb.log_evaluation(-1)])
    lgb_test_final  += np.expm1(m.predict(X_te_lgb))
    lgb_train_final += np.expm1(m.predict(X_tr_lgb))
    lgb_models.append(m)

lgb_test_final  /= len(SEEDS)
lgb_train_final /= len(SEEDS)

lgb_train_r2 = r2_score(yo_train, lgb_train_final)
lgb_test_r2  = r2_score(yo_test,  lgb_test_final)
print(f"\nLGB  Train R² = {lgb_train_r2:.4f}")
print(f"LGB  Test  R² = {lgb_test_r2:.4f}")
print(f"LGB  Test RMSE = {np.sqrt(mean_squared_error(yo_test, lgb_test_final)):.0f}")

fi_lgb = pd.Series(lgb_models[0].feature_importance('gain'), index=FEATURES)
fi_lgb = (fi_lgb / fi_lgb.sum() * 100).sort_values(ascending=False)


##  Cell 12 — CatBoost: 5-Fold CV

**Key anti-overfitting settings vs naive baseline:**
- `depth` 8 → **6**
- `l2_leaf_reg` 2 → **5**
- `min_data_in_leaf` 3 → **15**


In [ ]:
cat_idx = [X_train[FEATURES].columns.get_loc(c) for c in CAT_COLS]

print("=" * 60)
print("  CatBoost — 5-Fold CV (regularised)")
print("=" * 60)

best_iters_cb = []
cb_fold_r2    = []
X_tr_cb       = X_train[FEATURES].copy()
X_te_cb       = X_test[FEATURES].copy()

for fold, (tr, val) in enumerate(kf.split(X_tr_cb), 1):
    Xtr, Xval = X_tr_cb.iloc[tr], X_tr_cb.iloc[val]
    ytr, yval = y_train.iloc[tr],  y_train.iloc[val]
    yov        = yo_train.iloc[val]

    tp = Pool(Xtr, ytr, cat_features=cat_idx)
    vp = Pool(Xval, yval, cat_features=cat_idx)

    m  = CatBoostRegressor(
        iterations=2000, learning_rate=0.03,
        depth=6, l2_leaf_reg=5, min_data_in_leaf=15,
        subsample=0.80, colsample_bylevel=0.75,
        loss_function='RMSE', random_seed=42,
        verbose=0, early_stopping_rounds=150,
    )
    m.fit(tp, eval_set=vp)
    best_iters_cb.append(m.best_iteration_)

    p       = np.expm1(m.predict(Pool(Xval, cat_features=cat_idx)))
    fold_r2 = r2_score(yov, p)
    cb_fold_r2.append(fold_r2)
    rmse_fold = np.sqrt(mean_squared_error(yov, p))
    print(f"  Fold {fold}: RMSE={rmse_fold:>10.1f}  R²={fold_r2:.4f}  best_iter={m.best_iteration_}")

best_cb_n = int(np.mean(best_iters_cb))
print(f"\n  Mean CV R²: {np.mean(cb_fold_r2):.4f} ± {np.std(cb_fold_r2):.4f}")
print(f"  Mean best_iteration: {best_cb_n}")


## Cell 13 — CatBoost: Full-Train Refit

In [ ]:
print(f"Retraining on full train ({best_cb_n} iterations from CV)...")

final_cb = CatBoostRegressor(
    iterations=best_cb_n, learning_rate=0.03,
    depth=6, l2_leaf_reg=5, min_data_in_leaf=15,
    subsample=0.80, colsample_bylevel=0.75,
    loss_function='RMSE', random_seed=42, verbose=0,
)
final_cb.fit(Pool(X_tr_cb, y_train, cat_features=cat_idx))

cb_test_pred  = np.expm1(final_cb.predict(Pool(X_te_cb, cat_features=cat_idx)))
cb_train_pred = np.expm1(final_cb.predict(Pool(X_tr_cb, cat_features=cat_idx)))

cb_train_r2 = r2_score(yo_train, cb_train_pred)
cb_test_r2  = r2_score(yo_test,  cb_test_pred)
print(f"\nCB   Train R² = {cb_train_r2:.4f}")
print(f"CB   Test  R² = {cb_test_r2:.4f}")
print(f"CB   Test RMSE = {np.sqrt(mean_squared_error(yo_test, cb_test_pred)):.0f}")

fi_cb = pd.Series(final_cb.get_feature_importance(), index=X_tr_cb.columns)
fi_cb = (fi_cb / fi_cb.sum() * 100).sort_values(ascending=False)


##  Cell 14 — Final Ensemble & Metrics

In [ ]:
blend_test  = 0.5 * lgb_test_final  + 0.5 * cb_test_pred
blend_train = 0.5 * lgb_train_final + 0.5 * cb_train_pred

rmse  = np.sqrt(mean_squared_error(yo_test, blend_test))
mae   = mean_absolute_error(yo_test, blend_test)
r2    = r2_score(yo_test, blend_test)
r2_tr = r2_score(yo_train, blend_train)
gap   = r2_tr - r2

print("=" * 60)
print("  FINAL ENSEMBLE  (LightGBM×3seeds + CatBoost, 50/50)")
print("=" * 60)
print(f"  Train R²        : {r2_tr:.4f}")
print(f"  Test  R²        : {r2:.4f}")
print(f"  Train–Test Gap  : {gap:.4f}  (original naive: ~0.31)")
print(f"  Test  RMSE      : {rmse:.1f}  μmol/g·h")
print(f"  Test  MAE       : {mae:.1f}  μmol/g·h")
print(f"  Total features  : {len(FEATURES)}")


## Cell 15 — Plot: Parity (Observed vs Predicted)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
ax.scatter(yo_train, blend_train, alpha=0.35, s=18, color='steelblue')
lim = max(yo_train.max(), blend_train.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=1.5)
ax.set_title(f'Train  |  R² = {r2_tr:.4f}', fontsize=13, fontweight='bold')
ax.set_xlabel('Observed H₂ (μmol/g·h)'); ax.set_ylabel('Predicted H₂ (μmol/g·h)')

ax = axes[1]
ax.scatter(yo_test, blend_test, alpha=0.5, s=22, color='darkorange')
lim = max(yo_test.max(), blend_test.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=1.5)
ax.set_title(f'Test  |  R² = {r2:.4f}  RMSE = {rmse:.0f}', fontsize=13, fontweight='bold')
ax.set_xlabel('Observed H₂ (μmol/g·h)'); ax.set_ylabel('Predicted H₂ (μmol/g·h)')

plt.suptitle('LightGBM×3seeds + CatBoost Ensemble — H₂ Production Rate',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


##  Cell 16 — Plot: Overfitting Before vs After

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

labels = ['Original\n(Train)', 'Original\n(Test)', 'Fixed\n(Train)', 'Fixed\n(Test)']
values = [0.9903, 0.6762, r2_tr, r2]
colors = ['#d45f5f', '#e0a060', '#5a8fd4', '#5bc46a']

bars = ax.bar(labels, values, color=colors, edgecolor='white', width=0.5)
ax.axhline(0.80, color='gray', linestyle='--', lw=1.2, label='Theoretical ceiling ≈ 0.80')
ax.set_ylim(0, 1.05)
ax.set_ylabel('R²', fontsize=12)
ax.set_title('Overfitting Reduction: Original vs Fixed Model', fontsize=13, fontweight='bold')

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', fontsize=11, fontweight='bold')

ax.legend(fontsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


## 🔍 Cell 17 — Plot: Feature Importance (LightGBM)

In [ ]:
top20 = fi_lgb.head(20)

fig, ax = plt.subplots(figsize=(12, 7))
bars = ax.barh(top20.index[::-1], top20.values[::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Importance (%)', fontsize=12)
ax.set_title('Top 20 Feature Importances — LightGBM (Gain)', fontsize=13, fontweight='bold')
for bar, val in zip(bars, top20.values[::-1]):
    ax.text(bar.get_width() + 0.15, bar.get_y() + bar.get_height() / 2,
            f'{val:.1f}%', va='center', fontsize=9)
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


##  Cell 18 — Best Catalyst Analysis

In [ ]:
# Filter binning
filter_bins   = [-1, 350, 400, 420, 1000]
filter_labels = ['<350 nm', '350–400 nm', '400–420 nm', '>420 nm', 'No Filter']
df['filter_bin'] = pd.cut(df['Filter (nm)'], bins=filter_bins, labels=filter_labels[:-1])
df['filter_bin'] = df['filter_bin'].cat.add_categories(['No Filter'])
df.loc[no_filter_mask, 'filter_bin'] = 'No Filter'

# Top 10 observed
top10 = (df[[TARGET,'Cocatalyst 1','Semiconductor 1','Co-Catalyst wt','Bandgap (eV)','Power (W)','Filter (nm)']]
         .nlargest(10, TARGET).reset_index(drop=True))
top10.index += 1
print("── Top 10 Observed Experiments ──")
print(top10.to_string())

# Co-catalyst stats
cocat_stats = (df.groupby('Cocatalyst 1')[TARGET]
               .agg(mean='mean', median='median', max='max', count='count')
               .sort_values('mean', ascending=False).reset_index())
print("\n── Top Co-Catalysts (≥5 experiments) ──")
print(cocat_stats[cocat_stats['count'] >= 5].head(10).to_string(index=False))

# Semiconductor stats
sem1_stats = (df.groupby('Semiconductor 1')[TARGET]
              .agg(mean='mean', median='median', max='max', count='count')
              .sort_values('mean', ascending=False).reset_index())
print("\n── Top Semiconductors (≥5 experiments) ──")
print(sem1_stats[sem1_stats['count'] >= 5].head(10).to_string(index=False))

# Combos
combo = (df.groupby(['Cocatalyst 1','Semiconductor 1'])[TARGET]
         .agg(mean='mean', max='max', count='count').reset_index()
         .sort_values('mean', ascending=False))
combo_top = combo[combo['count'] >= 3].head(15).reset_index(drop=True)
combo_top.index += 1
print("\n── Top 15 Cocatalyst × Semiconductor Combos (≥3 experiments) ──")
print(combo_top.to_string())


## Cell 19 — Model-Predicted Best Catalyst Grid

In [ ]:
top_cocat = cocat_stats[cocat_stats['count'] >= 5].head(8)['Cocatalyst 1'].tolist()
top_sem1  = sem1_stats[sem1_stats['count'] >= 5].head(8)['Semiconductor 1'].tolist()
FILTER_SCENARIOS = {'No Filter (0)': 0, 'UV-Vis (365 nm)': 365, 'Visible (420 nm)': 420}
num_med  = {c: df[c].median() for c in NUM_COLS}
cat_def  = {c: df[c].mode()[0] for c in CAT_COLS}

def build_predict_grid(fval):
    rows = []
    for cc in top_cocat:
        for s1 in top_sem1:
            row = {**num_med, **cat_def}
            row['Cocatalyst 1']    = cc
            row['Semiconductor 1'] = s1
            row['Filter (nm)']     = fval
            rows.append(row)
    gdf = pd.DataFrame(rows)
    gdf['has_cocatalyst']     = (gdf['Cocatalyst 1'] != 'None').astype(int)
    gdf['has_semiconductor2'] = (gdf['Semiconductor 2'] != 'None').astype(int)
    gdf['has_filter']         = int(fval > 0)
    gdf['power_per_vol']      = gdf['Power (W)'] / (gdf['Solution Volume (ml)'] + 1)
    gdf['load_x_glycerol']    = gdf['Photocatalyst load (g/L)'] * gdf['Glycerol Concentration (v%)']
    gdf['bandgap_x_power']    = gdf['Bandgap (eV)'] * gdf['Power (W)']
    gdf['wt_per_load']        = gdf['Co-Catalyst wt'] / (gdf['Photocatalyst load (g/L)'] + 1e-3)
    gdf['log_power']          = np.log1p(gdf['Power (W)'])
    gdf['log_load']           = np.log1p(gdf['Photocatalyst load (g/L)'])
    for col in TE_COLS:
        feat   = f'te_{col}'
        te_map = y_train.groupby(X_train[col]).mean()
        gdf[feat] = gdf[col].map(te_map).fillna(y_train.mean())
    return gdf

all_pivots = {}
for scenario, fval in FILTER_SCENARIOS.items():
    gdf    = build_predict_grid(fval)
    Xg     = gdf[FEATURES].copy()
    Xg_lgb = Xg.copy()
    for c in CAT_COLS: Xg_lgb[c] = Xg_lgb[c].astype('category')
    g_lgb = np.mean([np.expm1(m.predict(Xg_lgb)) for m in lgb_models], axis=0)
    g_cb  = np.expm1(final_cb.predict(Pool(Xg, cat_features=cat_idx)))
    gdf['pred_H2'] = 0.5 * g_lgb + 0.5 * g_cb
    pivot = gdf.pivot_table(index='Cocatalyst 1', columns='Semiconductor 1',
                             values='pred_H2', aggfunc='mean')
    all_pivots[scenario] = pivot
    bi = gdf['pred_H2'].idxmax()
    print(f"[{scenario}]")
    print(f"  ★ Best: Cocatalyst={gdf.loc[bi,'Cocatalyst 1']}  "
          f"Semiconductor={gdf.loc[bi,'Semiconductor 1']}  →  H₂ ≈ {gdf.loc[bi,'pred_H2']:.0f} μmol/g·h\n")


##  Cell 20 — Plot: Best Catalyst Analysis (4-panel)

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.4)

ax1 = fig.add_subplot(gs[0, 0])
top_cc = cocat_stats[cocat_stats['count'] >= 5].head(10)
bars = ax1.barh(top_cc['Cocatalyst 1'][::-1], top_cc['mean'][::-1], color='steelblue', edgecolor='white')
ax1.set_xlabel('Mean H₂ Rate (μmol/g·h)', fontsize=10)
ax1.set_title('Top Co-Catalysts (mean H₂, ≥5 exp)', fontsize=11, fontweight='bold')
for bar, val in zip(bars, top_cc['mean'][::-1]):
    ax1.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2, f'{val:.0f}', va='center', fontsize=8)
ax1.grid(axis='x', linestyle='--', alpha=0.4)

ax2 = fig.add_subplot(gs[0, 1])
top_s1 = sem1_stats[sem1_stats['count'] >= 5].head(10)
bars = ax2.barh(top_s1['Semiconductor 1'][::-1], top_s1['mean'][::-1], color='darkorange', edgecolor='white')
ax2.set_xlabel('Mean H₂ Rate (μmol/g·h)', fontsize=10)
ax2.set_title('Top Semiconductors (mean H₂, ≥5 exp)', fontsize=11, fontweight='bold')
for bar, val in zip(bars, top_s1['mean'][::-1]):
    ax2.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2, f'{val:.0f}', va='center', fontsize=8)
ax2.grid(axis='x', linestyle='--', alpha=0.4)

ax3 = fig.add_subplot(gs[1, 0])
labels_combo = [f"{r['Cocatalyst 1']}/{r['Semiconductor 1']}" for _, r in combo_top.iterrows()]
values_combo = combo_top['mean'].values
colors_combo = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(labels_combo)))
bars = ax3.barh(labels_combo[::-1], values_combo[::-1], color=colors_combo, edgecolor='white')
ax3.set_xlabel('Mean H₂ Rate (μmol/g·h)', fontsize=10)
ax3.set_title('Top 15 Cocatalyst×Semiconductor Combos (≥3 exp)', fontsize=11, fontweight='bold')
ax3.tick_params(axis='y', labelsize=8)
for bar, val in zip(bars, values_combo[::-1]):
    ax3.text(bar.get_width()*1.01, bar.get_y()+bar.get_height()/2, f'{val:.0f}', va='center', fontsize=7)
ax3.grid(axis='x', linestyle='--', alpha=0.4)

ax4 = fig.add_subplot(gs[1, 1])
pv = all_pivots['No Filter (0)'].fillna(0)
im = ax4.imshow(pv.values, aspect='auto', cmap='YlOrRd')
ax4.set_xticks(range(len(pv.columns))); ax4.set_xticklabels(pv.columns, rotation=45, ha='right', fontsize=8)
ax4.set_yticks(range(len(pv.index)));   ax4.set_yticklabels(pv.index, fontsize=8)
ax4.set_title('Predicted H₂ Heatmap — No Filter', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax4, label='Predicted H₂ (μmol/g·h)', shrink=0.8)
for i in range(len(pv.index)):
    for j in range(len(pv.columns)):
        v = pv.values[i, j]
        ax4.text(j, i, f'{v:.0f}', ha='center', va='center',
                 fontsize=6, color='black' if v < pv.values.max()*0.6 else 'white')

fig.suptitle('Best Catalyst Analysis — H₂ Photocatalytic Production', fontsize=15, y=1.01, fontweight='bold')
plt.show()


##  Cell 21 — Plot: Residual Analysis

In [ ]:
residuals = yo_test.values - blend_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(blend_test, residuals, alpha=0.5, s=20, color='steelblue')
ax.axhline(0, color='r', linestyle='--', lw=1.5)
ax.set_xlabel('Predicted H₂'); ax.set_ylabel('Residual')
ax.set_title('Residuals vs Predicted', fontsize=12, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)

ax = axes[1]
ax.hist(residuals, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(0, color='r', linestyle='--', lw=1.5)
ax.set_xlabel('Residual (μmol/g·h)'); ax.set_ylabel('Count')
ax.set_title('Residual Distribution', fontsize=12, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.4)

plt.suptitle('Residual Analysis — Ensemble Model', fontsize=13)
plt.tight_layout()
plt.show()


##  Cell 22 — Plot: Filter Wavelength Analysis

In [ ]:
filter_stats = (df.groupby('filter_bin', observed=False)[TARGET]
                .agg(mean='mean', median='median', max='max', count='count')
                .dropna().sort_values('mean', ascending=False).reset_index())

fig = plt.figure(figsize=(18, 12))
gs5 = gridspec.GridSpec(2, 3, figure=fig, hspace=0.5, wspace=0.4)

ax_a = fig.add_subplot(gs5[0, 0])
fc    = filter_stats.copy()
bar_c = ['#4a9aba' if str(fb) == 'No Filter' else '#e07b39' for fb in fc['filter_bin']]
bars  = ax_a.bar(fc['filter_bin'].astype(str), fc['mean'], color=bar_c, edgecolor='white', width=0.6)
ax_a.set_title('Mean H₂ by Filter Bin', fontsize=11, fontweight='bold')
ax_a.set_xlabel('Filter Range'); ax_a.set_ylabel('Mean H₂ (μmol/g·h)')
ax_a.tick_params(axis='x', rotation=20, labelsize=8)
for bar, val in zip(bars, fc['mean']):
    ax_a.text(bar.get_x()+bar.get_width()/2, bar.get_height()*1.02, f'{val:.0f}', ha='center', fontsize=8)
ax_a.grid(axis='y', linestyle='--', alpha=0.4)

ax_b = fig.add_subplot(gs5[0, 1])
grps  = [df[df['filter_bin']==lb][TARGET].dropna().values for lb in filter_labels if len(df[df['filter_bin']==lb]) > 0]
glbls = [lb for lb in filter_labels if len(df[df['filter_bin']==lb]) > 0]
bp    = ax_b.boxplot(grps, labels=glbls, patch_artist=True, showfliers=True,
                     flierprops=dict(marker='o', markersize=3, alpha=0.4))
for patch in bp['boxes']: patch.set_facecolor('#e07b39'); patch.set_alpha(0.7)
ax_b.set_title('H₂ Distribution by Filter Bin', fontsize=11, fontweight='bold')
ax_b.tick_params(axis='x', rotation=20, labelsize=8)
ax_b.grid(axis='y', linestyle='--', alpha=0.4)

ax_c = fig.add_subplot(gs5[0, 2])
df_f = df[df['Filter (nm)'] > 0]
sc   = ax_c.scatter(df_f['Filter (nm)'], df_f[TARGET], c=df_f['Power (W)'], cmap='viridis', alpha=0.5, s=20)
plt.colorbar(sc, ax=ax_c, label='Power (W)', shrink=0.85)
ax_c.set_xlabel('Filter (nm)'); ax_c.set_ylabel('H₂ Rate (μmol/g·h)')
ax_c.set_title('Filter vs H₂ Rate (colour=Power)', fontsize=11, fontweight='bold')
ax_c.grid(True, linestyle='--', alpha=0.4)

for ax_idx, scenario in zip([gs5[1,0], gs5[1,1]], ['UV-Vis (365 nm)', 'Visible (420 nm)']):
    ax_s  = fig.add_subplot(ax_idx)
    pdata = all_pivots[scenario].fillna(0)
    im_s  = ax_s.imshow(pdata.values, aspect='auto', cmap='YlOrRd')
    ax_s.set_xticks(range(len(pdata.columns))); ax_s.set_xticklabels(pdata.columns, rotation=45, ha='right', fontsize=7)
    ax_s.set_yticks(range(len(pdata.index)));   ax_s.set_yticklabels(pdata.index, fontsize=7)
    ax_s.set_title(f'Predicted H₂ — {scenario}', fontsize=11, fontweight='bold')
    plt.colorbar(im_s, ax=ax_s, label='H₂ (μmol/g·h)', shrink=0.8)
    for i in range(len(pdata.index)):
        for j in range(len(pdata.columns)):
            v = pdata.values[i, j]
            ax_s.text(j, i, f'{v:.0f}', ha='center', va='center',
                      fontsize=5, color='black' if v < pdata.values.max()*0.6 else 'white')

ax_f = fig.add_subplot(gs5[1, 2])
cocat_filter = (df.groupby(['Cocatalyst 1','filter_bin'], observed=False)[TARGET]
                .agg(mean='mean').reset_index().dropna())
cfp   = cocat_filter.pivot_table(index='Cocatalyst 1', columns='filter_bin', values='mean', aggfunc='mean')
valid = cocat_stats[cocat_stats['count'] >= 5]['Cocatalyst 1'].tolist()
cfp   = cfp.loc[cfp.index.isin(valid)]
im_f  = ax_f.imshow(cfp.fillna(0).values, aspect='auto', cmap='Blues')
ax_f.set_xticks(range(len(cfp.columns))); ax_f.set_xticklabels([str(c) for c in cfp.columns], rotation=30, ha='right', fontsize=7)
ax_f.set_yticks(range(len(cfp.index)));   ax_f.set_yticklabels(cfp.index, fontsize=7)
ax_f.set_title('Observed H₂: Cocatalyst × Filter Bin', fontsize=11, fontweight='bold')
plt.colorbar(im_f, ax=ax_f, label='Mean H₂ (μmol/g·h)', shrink=0.8)
for i in range(cfp.shape[0]):
    for j in range(cfp.shape[1]):
        v = cfp.fillna(0).values[i, j]
        if v > 0:
            ax_f.text(j, i, f'{v:.0f}', ha='center', va='center',
                      fontsize=5, color='black' if v < cfp.values.max()*0.6 else 'white')

fig.suptitle('Filter (nm) Analysis — H₂ Photocatalytic Production', fontsize=15, y=1.01, fontweight='bold')
plt.show()


## Cell 23 — Final Summary

In [ ]:
print("=" * 60)
print("  FINAL SUMMARY")
print("=" * 60)
print(f"  Models         : LightGBM (×{len(SEEDS)} seeds, 5-Fold CV) + CatBoost (5-Fold CV)")
print(f"  Ensemble       : Simple 50/50 average")
print(f"  Total features : {len(FEATURES)}")
print(f"  Train R²       : {r2_tr:.4f}")
print(f"  Test  R²       : {r2:.4f}")
print(f"  Train–Test Gap : {gap:.4f}  (original naive: ~0.31)")
print(f"  Test  RMSE     : {rmse:.1f}  μmol/g·h")
print(f"  Test  MAE      : {mae:.1f}  μmol/g·h")

print(f"\n  Overfitting fixes applied:")
fixes = [
    ("num_leaves (LGB)",        "200",   "64"),
    ("min_data_in_leaf",        "3",     "20"),
    ("lambda_l2 (LGB)",         "0.05",  "1.0"),
    ("Full-train boost rounds", "4000 fixed", "CV best_iter"),
    ("Combo cat features",      "3 cols (up to 806 cats)", "Removed"),
    ("Engineered features",     "22",    "9"),
    ("CatBoost depth",          "8",     "6"),
    ("CatBoost l2_leaf_reg",    "2",     "5"),
    ("Target encoding scope",   "5 cols incl. combos", "2 core cols only"),
]
print(f"  {'Fix':<30} {'Before':<25} {'After'}")
print("  " + "-"*65)
for name, before, after in fixes:
    print(f"  {name:<30} {before:<25} {after}")

print(f"\n  ★ Best observed experiment ★")
best = df.loc[df[TARGET].idxmax()]
print(f"    Cocatalyst   : {best['Cocatalyst 1']}")
print(f"    Semiconductor: {best['Semiconductor 1']}")
print(f"    H₂ Rate      : {best[TARGET]:.1f} μmol/g·h")

print(f"\n  ★ Best model-predicted combinations ★")
for scenario, fval in FILTER_SCENARIOS.items():
    gdf    = build_predict_grid(fval)
    Xg     = gdf[FEATURES].copy()
    Xg_lgb = Xg.copy()
    for c in CAT_COLS: Xg_lgb[c] = Xg_lgb[c].astype('category')
    g_lgb = np.mean([np.expm1(m.predict(Xg_lgb)) for m in lgb_models], axis=0)
    g_cb  = np.expm1(final_cb.predict(Pool(Xg, cat_features=cat_idx)))
    gdf['pred_H2'] = 0.5*g_lgb + 0.5*g_cb
    bi = gdf['pred_H2'].idxmax()
    print(f"    [{scenario}] {gdf.loc[bi,'Cocatalyst 1']} + {gdf.loc[bi,'Semiconductor 1']} → {gdf.loc[bi,'pred_H2']:.0f} μmol/g·h")
print("=" * 60)
